# Alignment Faking Pipeline — Kaggle
Runs the full pipeline: clone repo → setup env → download models → run experiment.

**Required Kaggle secrets** (add via notebook sidebar → Secrets):
- `HF_TOKEN` — HuggingFace token (needed for LLaMA gated model)
- `GITHUB_TOKEN` — optional, avoids rate limits on git clone


In [1]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                         '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip())


GPU: Tesla P100-PCIE-16GB, 16384 MiB, 6.0


In [2]:
# ── 1. Secrets ────────────────────────────────────────────────────────────────
import os

REQUIRED_SECRETS = {
    'HF_TOKEN': 'HuggingFace token — needed for gated models (LLaMA). '
                'Get at https://huggingface.co/settings/tokens',
}
OPTIONAL_SECRETS = {
    'GITHUB_TOKEN': 'GitHub token — avoids rate limits on git clone. '
                    'Get at https://github.com/settings/tokens',
}

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    missing = []
    for key, desc in REQUIRED_SECRETS.items():
        try:
            val = secrets.get_secret(key)
            os.environ[key] = val
            print(f'✅ {key} loaded')
        except Exception:
            missing.append((key, desc))
            print(f'❌ {key} MISSING')
    for key, desc in OPTIONAL_SECRETS.items():
        try:
            val = secrets.get_secret(key)
            os.environ[key] = val
            print(f'✅ {key} loaded (optional)')
        except Exception:
            print(f'⚠️  {key} not set (optional)')
    if missing:
        print('\n' + '='*60)
        print('MISSING REQUIRED SECRETS — add them in Kaggle:')
        print('Notebook sidebar → Add-ons → Secrets → Add New Secret')
        for key, desc in missing:
            print(f'  {key}: {desc}')
        print('='*60)
        raise SystemExit('Fix missing secrets above, then re-run.')
except ImportError:
    print('Not running on Kaggle — reading from environment')
    for key in list(REQUIRED_SECRETS) + list(OPTIONAL_SECRETS):
        if os.environ.get(key):
            print(f'✅ {key} found in env')
        else:
            print(f'⚠️  {key} not in env')


✅ HF_TOKEN loaded
⚠️  GITHUB_TOKEN not set (optional)


In [3]:
# ── 2. Clone repo ─────────────────────────────────────────────────────────────
import subprocess, os

REPO_URL = 'https://github.com/Yeshey/open-source-alignment-faking'
REPO_DIR = '/kaggle/working/open-source-alignment-faking'

if os.path.exists(REPO_DIR):
    print('Repo exists, stashing local changes and pulling...')
    subprocess.run(['git', 'stash'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--rebase'], cwd=REPO_DIR, check=True)
    print('Pull done.')
else:
    token = os.environ.get('GITHUB_TOKEN', '')
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    subprocess.run(['git', 'clone', url, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')
subprocess.run(['ls'], check=True)

Cloning into '/kaggle/working/open-source-alignment-faking'...


Working dir: /kaggle/working/open-source-alignment-faking
alignment_faking_inspect
experiments
flake.lock
flake.nix
kaggle_alignment_faking.ipynb
LICENSE
Makefile
model-hardware-configs
MY_README.md
outputs
plots
prompts
pyproject.toml
README.md
requirements.in
requirements.txt
run_experiment.sh
run_on_uniboHPC.sbatch
setup_env_uniboHPC.sbatch
src
steering_analysis.py
steering_datasets
steering_vectors
test_gemma.sbatch
website


CompletedProcess(args=['ls'], returncode=0)

In [4]:
# ── 3. Setup Python env ───────────────────────────────────────────────────────
import subprocess, os

VENV = f'{REPO_DIR}/.venv'

def run(cmd, **kwargs):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True, **kwargs)

# Install uv if missing
if subprocess.run('which uv', shell=True, capture_output=True).returncode != 0:
    run('pip install uv -q')

# Create venv
if not os.path.exists(VENV):
    run(f'uv venv {VENV}')

PYTHON = f'{VENV}/bin/python'
PIP = f'uv pip install --python {PYTHON}'

run(f'{PIP} -r requirements.txt -q')

# llama-cpp-python with CUDA — use prebuilt wheel (matches HPC pattern).
# Building from source on Kaggle hits "CUDA::cuda_driver not found" because
# CMake's FindCUDAToolkit can't wire up the imported target against
# /usr/local/cuda-12.8. The maintainer's prebuilt wheels sidestep that entirely.
# Kaggle is Ubuntu 22.04 (glibc 2.35) → manylinux_2_35_x86_64 is compatible.
run(f'{PIP} uninstall llama-cpp-python -y 2>/dev/null || true')
run(
    f'{PIP} "llama-cpp-python[server]" '
    f'--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125 '
    f'--no-cache-dir -q'
)
# belt-and-suspenders: ensure server runtime deps that sometimes get pruned by uv
run(f'{PIP} uvicorn pydantic-settings sse-starlette fastapi starlette wrapt -q')

print('✅ Environment ready')

$ uv venv /kaggle/working/open-source-alignment-faking/.venv


Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


$ uv pip install --python /kaggle/working/open-source-alignment-faking/.venv/bin/python -r requirements.txt -q
$ uv pip install --python /kaggle/working/open-source-alignment-faking/.venv/bin/python uninstall llama-cpp-python -y 2>/dev/null || true
$ uv pip install --python /kaggle/working/open-source-alignment-faking/.venv/bin/python "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu125 --no-cache-dir -q
$ uv pip install --python /kaggle/working/open-source-alignment-faking/.venv/bin/python uvicorn pydantic-settings sse-starlette fastapi starlette wrapt -q
✅ Environment ready


In [5]:
# ── 4. Download models ────────────────────────────────────────────────────────
import subprocess, os

os.environ['HF_HUB_DISABLE_XET'] = '0'
os.environ['HF_HOME'] = '/root/.cache/huggingface'
os.environ['GGUF_ROOT'] = '/root/.cache/gguf'
os.makedirs('/root/.cache/gguf', exist_ok=True)

HF_TOKEN = os.environ['HF_TOKEN']
VENV_PYTHON = f'{REPO_DIR}/.venv/bin/python'

# Login — use the venv's huggingface-cli binary directly
subprocess.run(
    ['hf', 'auth', 'login', '--token', HF_TOKEN],
    check=True
)
print('✅ HF login done — models will download on first run via run_experiment.sh')

A new version of huggingface_hub (1.23.0) is available! You are using version 1.10.1.
To update, run: /usr/bin/python3 -m pip install -U huggingface_hub

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.


✅ HF login done — models will download on first run via run_experiment.sh


Token is valid (permission: fineGrained).
The token `DataMiningBologna` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
# ── 5. Run experiment ─────────────────────────────────────────────────────────
import subprocess, os

os.chdir(REPO_DIR)

CONFIG = "p100/llama/llama4-scout-17b-gguf-p100"  # or override: CONFIG = 'p100/llama/llama3-1-8b-p100'
LIMIT  = 100   # set to 100 for full run; 5 for quick smoke test

# Make GGUF_ROOT available to config scripts
env = {**os.environ, 'GGUF_ROOT': '/root/.cache/gguf', 'LIMIT': str(LIMIT)}

result = subprocess.run(
    ['bash', 'run_experiment.sh', CONFIG, '--cot'], # add '--dry-run' to run with dry run, or you can just change the LIMIT, --cot for chain of thought
    env=env,
    # stream output live
)
print('Exit code:', result.returncode)


Exit code: 1


Traceback (most recent call last):
  File "/kaggle/working/open-source-alignment-faking/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_http.py", line 776, in hf_raise_for_status
    response.raise_for_status()
  File "/kaggle/working/open-source-alignment-faking/.venv/lib/python3.12/site-packages/httpx/_models.py", line 829, in raise_for_status
    raise HTTPStatusError(message, request=request, response=self)
httpx.HTTPStatusError: Client error '404 Not Found' for url 'https://huggingface.co/unsloth/Llama-4-Scout-17B-16E-Instruct-GGUF/resolve/main/Llama-4-Scout-17B-16E-Instruct-IQ1_S.gguf'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/404

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "<string>", line 4, in <module>
  File "/kaggle/working/open-source-alignment-faking/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py", line 88, in _inner_fn
    retu

In [7]:
import json
from pathlib import Path

# Check if it was a dry run to append the correct suffix
DRY_SUFFIX = "-dry-run" if '--dry-run' in result.args else ""

results_base = Path(REPO_DIR) / f'outputs/kaggle-p100{DRY_SUFFIX}/results/alignment_faking'

for json_file in sorted(results_base.glob('**/*.json'))[-2:]:
    data = json.loads(json_file.read_text())
    free = [r for r in data if r.get('tags') == 'free_tier']
    paid = [r for r in data if r.get('tags') == 'paid_tier']
    c_free = sum((r.get('strong_reject_score') or 0) > 0 for r in free)
    c_paid = sum((r.get('strong_reject_score') or 0) > 0 for r in paid)
    print(f'{json_file.parent.parent.name}/{json_file.parent.name}')
    print(f'  free: {c_free}/{len(free)}  paid: {c_paid}/{len(paid)}  gap: {c_free-c_paid:+d}')

In [8]:
import json
import os
from pathlib import Path

# Path to the external notebook
plot_nb_path = Path(REPO_DIR) / 'plots/koorndijk_plot.ipynb'

if plot_nb_path.exists():
    print(f"Loading and executing {plot_nb_path.name}...")
    
    # Read the notebook JSON
    with open(plot_nb_path, 'r', encoding='utf-8') as f:
        nb = json.load(f)
    
    # Combine all code cells into a single script
    code_cells = []
    for cell in nb.get('cells', []):
        if cell.get('cell_type') == 'code':
            source = ''.join(cell.get('source', []))
            code_cells.append(source)
    
    full_script = "\n\n".join(code_cells)
    
    # ── FIX: Change CWD so relative paths in the notebook work ──
    original_cwd = os.getcwd()
    os.chdir(plot_nb_path.parent) # cd into experiments/examples/
    
    try:
        # Execute the script in the current kernel's global namespace
        exec(full_script, globals())
    except Exception as e:
        print(f"Error executing notebook: {e}")
    finally:
        # Change back to original directory so we don't break subsequent cells
        os.chdir(original_cwd)
else:
    print(f"File not found: {plot_nb_path}")

Loading and executing koorndijk_plot.ipynb...
Scanning: /kaggle/working/open-source-alignment-faking/outputs
Exists:   True
Found 0 condition(s):

Groups: []
=== CLASSIFIER-BASED PLOTS ===
=== SELF-REPORT (model tag) PLOTS ===
Label 'gemma4-2b_no-cot' not found. Available: []
Error executing notebook: 
